# 01. Аудит исходных данных H&M

Ноутбук только читает сохранённые артефакты `hmrp audit` и `hmrp build-features` (`reports/data/`). Пересчёт: `hmrp audit --config configs/full.yaml`.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 160, "display.max_columns", 30)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
REP = ROOT / "reports"
def read(p): return json.loads(Path(p).read_text(encoding="utf-8"))

In [ ]:
a = read(REP / "data" / "source_audit.json")
pd.DataFrame(a["loads"])[["file_role", "file_name", "load_id", "rows_read", "rows_loaded", "rows_rejected_structure", "rows_rejected_type", "size_bytes", "sha256"]]

In [ ]:
pd.Series(a["transactions"]).to_frame("transactions")

## Цена (предоставленная шкала) и совпадающие строки

In [ ]:
pr = a["price"]
pd.DataFrame({"level": pr["quantile_levels"], "price": pr["quantiles"]}).set_index("level").T.assign(n_zero=pr["n_zero"], n_negative=pr["n_negative"])

In [ ]:
pd.DataFrame(a["identical_rows"]).head(10)

## Динамика по дням и каналам

In [ ]:
daily = pd.read_csv(REP / "data" / "source_daily_by_channel.csv", parse_dates=["t_dat"])
w = daily.pivot_table(index="t_dat", columns="sales_channel_id", values="n_rows", aggfunc="sum").resample("W").sum()
ax = w.plot(figsize=(10, 3.5), linewidth=2, title="Item rows per week by sales_channel_id"); ax.set_ylabel("rows")

In [ ]:
m = pd.read_csv(REP / "data" / "source_monthly_by_channel.csv")
share = m.pivot_table(index="month", columns="sales_channel_id", values="n_rows", aggfunc="sum")
(share.div(share.sum(axis=1), axis=0) * 100).round(1)

## Покрытие ключей и проверки JOIN

In [ ]:
pd.DataFrame(a["coverage"]).T

In [ ]:
pd.DataFrame({k: {kk: vv for kk, vv in v.items() if kk not in ("dim_key",)} for k, v in a["join_checks"].items()}).T

In [ ]:
read(REP / "data" / "join_trap_demo.json")  # demonstration on synthetic rows, not an H&M finding

## Клиентская витрина (full)

In [ ]:
idx = read(REP / "runs" / "index.json")
run = read(REP / "runs" / idx["full"] / "selection_decision.json")
ds = read(REP / "data" / run["dataset_id"] / "dataset_summary.json")
pd.DataFrame(ds["cohort_by_cutoff"])

In [ ]:
pd.Series(ds["checks"]["customer_day_reconciliation"]).to_frame("value")

In [ ]:
pd.Series(ds["purchase_days_per_customer"]["histogram_capped_50"]).head(15)

In [ ]:
pd.Series({k: v for k, v in ds["coverage"].items() if k != "missing_dates"}).to_frame("coverage")